In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
from scipy.special import expit


In [ ]:
data = np.load("data/a9a_l2_problem.npz")
X = data["X"]
y = data["y"]
L0 = float(data["L0"])
L1 = float(data["L1"])
mu = float(data["mu"])
x_star = data["x_star"]
f_star = float(data["f_star"])
lambda_reg = float(data["lambda_reg"])

print(X.shape, L0, L1, mu, f_star)


In [ ]:
delta = 1e-4
K = 4.0
budget = 10_000
start_gap = 3.0
x0_seed = 7
noise_seed = 0
c_R = 1.0
relax_iters = 18
stm_restart_c = 8.0
y_floor = 1e-9


In [ ]:
def f(w):
    yz = y * (X @ w)
    return float(np.logaddexp(0.0, -yz).mean() + 0.5 * lambda_reg * float(w @ w))


def grad(w):
    yz = y * (X @ w)
    s = expit(-yz)
    return -(X.T @ (y * s)) / X.shape[0] + lambda_reg * w


class Problem:
    def __init__(self, f, grad, L0, L1, mu, x_star, f_star):
        self.f = f
        self.grad = grad
        self.L0 = L0
        self.L1 = L1
        self.mu = mu
        self.x_star = x_star
        self.f_star = f_star


problem = Problem(f, grad, L0, L1, mu, x_star, f_star)


In [ ]:
nu = 0.5
for _ in range(200):
    nu = np.exp(-nu)


def gap_value(prob, x):
    return float(prob.f(x) - prob.f_star)


def pick_start(prob, target, seed):
    rng = np.random.default_rng(seed)
    u = rng.normal(size=prob.x_star.shape)
    u = u / np.linalg.norm(u)
    left, right = 0.0, 20.0
    for _ in range(60):
        middle = 0.5 * (left + right)
        if gap_value(prob, prob.x_star + middle * u) < target:
            left = middle
        else:
            right = middle
    return prob.x_star + 0.5 * (left + right) * u


def random_noise_unit(dim, rng):
    u = rng.normal(size=dim)
    return u / np.linalg.norm(u)


def threshold_alpha(K):
    return 1.0 / (K - 1.0)


def proj_ball(x, center, radius):
    r = x - center
    nr = np.linalg.norm(r)
    if nr <= radius:
        return x
    return center + radius * r / nr


def agmsdr_constants(K, c_R):
    alpha = threshold_alpha(K)
    kappa = (1.0 - alpha) ** 2 / (4.0 * (1.0 + alpha) ** 2)
    C_M = (1.0 + alpha) ** 2 / (2.0 * kappa)
    C_st = 3.0 * C_M / kappa
    c_K = max((16.0 * C_st) ** 0.5, (16.0 * C_st) ** (1.0 / 3.0))
    eps_coef = max((K + 1.0) ** 2 / 2.0, 128.0 * (c_R + 2.0) ** 2)
    return c_K, C_st, eps_coef


In [ ]:
def segment_min_beta(v, x, iters=18):
    d = x - v
    Xv = X @ v
    Xd = X @ d
    vd = float(v @ d)
    dd = float(d @ d)

    def dphi(beta):
        yz = y * (Xv + beta * Xd)
        data_der = float(np.mean(-y * Xd * expit(-yz)))
        reg_der = lambda_reg * (vd + beta * dd)
        return data_der + reg_der

    if dphi(0.0) >= 0.0:
        return 0.0
    if dphi(1.0) <= 0.0:
        return 1.0

    left, right = 0.0, 1.0
    for _ in range(iters):
        middle = 0.5 * (left + right)
        if dphi(middle) <= 0.0:
            left = middle
        else:
            right = middle
    return 0.5 * (left + right)


In [ ]:
def run_clipped_gd(prob, x0, delta, K, noise_seed, budget):
    rng = np.random.default_rng(noise_seed)
    alpha = threshold_alpha(K)
    x = x0.copy()
    gaps = [gap_value(prob, x)]
    marker_call = None
    marker_gap = None

    for k in range(budget):
        gt = prob.grad(x) + delta * random_noise_unit(x.size, rng)
        ngt = np.linalg.norm(gt)
        if marker_call is None and ngt <= K * delta:
            marker_call = k + 1
            marker_gap = gaps[-1]
        h = 1.0 / (2.0 * (1.0 + alpha))
        h = h / (prob.L0 + prob.L1 * ngt / (1.0 - alpha))
        x = x - h * gt
        gaps.append(gap_value(prob, x))

    return {"gaps": np.asarray(gaps), "marker_call": marker_call, "marker_gap": marker_gap}


def run_restart_stm(prob, x0, delta, K, noise_seed, budget, c=8.0):
    rng = np.random.default_rng(noise_seed)
    eta = nu / 2.0
    Ns = int(np.ceil(c * np.sqrt(prob.L0 / prob.mu)))
    yk = x0.copy()
    gaps = [gap_value(prob, yk)]
    calls = 0
    marker_call = None
    marker_gap = None

    while calls < budget:
        zk = yk.copy()
        A = 0.0
        G = prob.L0
        for k in range(min(Ns, budget - calls)):
            a = eta * (k + 2) / 2.0
            A_next = A + a
            xk = (A * yk + a * zk) / A_next
            gt = prob.grad(xk) + delta * random_noise_unit(xk.size, rng)
            ngt = np.linalg.norm(gt)
            calls += 1
            if marker_call is None and ngt <= K * delta:
                marker_call = calls
                marker_gap = gap_value(prob, xk)
            G = max(G, prob.L0 + prob.L1 * ngt)
            zk = zk - a * gt / G
            yk = (A * yk + a * zk) / A_next
            A = A_next
            gaps.append(gap_value(prob, yk))

    return {"gaps": np.asarray(gaps), "marker_call": marker_call, "marker_gap": marker_gap, "Ns": Ns}


In [ ]:
def agmsdr_stage(prob, u, Q_center, Q_radius, K, delta, N, rng, calls0, relax_iters):
    alpha = threshold_alpha(K)
    x = u.copy()
    v = u.copy()
    A = 0.0
    s = np.zeros_like(u)
    gaps = [gap_value(prob, u)]
    marker_call = None
    marker_gap = None

    for k in range(N):
        beta = segment_min_beta(v, x, iters=relax_iters)
        yk = v + beta * (x - v)
        gt = prob.grad(yk) + delta * random_noise_unit(yk.size, rng)
        ngt = np.linalg.norm(gt)
        call = calls0 + k + 1
        if marker_call is None and ngt <= K * delta:
            marker_call = call
            marker_gap = gap_value(prob, yk)

        h = 1.0 / (2.0 * (1.0 + alpha))
        h = h / (prob.L0 + prob.L1 * ngt / (1.0 - alpha))
        x_next = yk - h * gt
        decrease = prob.f(yk) - prob.f(x_next)
        if decrease <= 0.0 or not np.isfinite(decrease):
            decrease = 1e-18

        M = max(ngt ** 2 / (2.0 * decrease), 1e-300)
        a = (1.0 + np.sqrt(1.0 + 4.0 * M * A)) / (2.0 * M)
        A = A + a
        s = s + a * gt
        v = proj_ball(u - s, Q_center, Q_radius)
        x = x_next
        gaps.append(gap_value(prob, x))

    return x, np.asarray(gaps), marker_call, marker_gap


def run_agmsdr(prob, x0, delta, K, c_R, noise_seed, budget, relax_iters=18):
    rng = np.random.default_rng(noise_seed)
    c_K, _, _ = agmsdr_constants(K, c_R)
    xs = x0.copy()
    gaps = [gap_value(prob, xs)]
    calls = 0
    marker_call = None
    marker_gap = None

    while calls < budget:
        Fs = max(gap_value(prob, xs), 0.0)
        Rs = np.sqrt(2.0 * Fs / prob.mu)
        Ns = int(np.ceil(c_K * (np.sqrt(prob.L0 / prob.mu) + (prob.L1 ** 2 * Fs / prob.mu) ** (1.0 / 3.0))))
        Ns = max(2, min(Ns, budget - calls))
        xs, stage_gaps, mc, mg = agmsdr_stage(prob, xs, xs, c_R * Rs, K, delta, Ns, rng, calls, relax_iters)
        gaps.extend(stage_gaps[1:].tolist())
        calls += len(stage_gaps) - 1
        if marker_call is None and mc is not None:
            marker_call = mc
            marker_gap = mg

    return {"gaps": np.asarray(gaps[:budget + 1]), "marker_call": marker_call, "marker_gap": marker_gap, "c_K": c_K}


In [ ]:
x0 = pick_start(problem, start_gap, x0_seed)
F0 = gap_value(problem, x0)
c_K, C_st, eps_coef = agmsdr_constants(K, c_R)

print(f"a9a: n,d={X.shape}, L0={L0:.4f}, L1={L1:.4f}, mu={mu:.3e}, L0/mu={L0 / mu:.0f}")
print(f"F0={F0:.3f}, delta={delta:g}, K={K:g}, budget={budget}")
print(f"AGMsDR c_K={c_K:.3f}, C_st={C_st:.1f}, floor coefficient={eps_coef:.0f}")

agmsdr = run_agmsdr(problem, x0, delta, K, c_R, noise_seed, budget, relax_iters)
restart_stm = run_restart_stm(problem, x0, delta, K, noise_seed, budget, stm_restart_c)
clipped_gd = run_clipped_gd(problem, x0, delta, K, noise_seed, budget)

print("Projected Restarted AGMsDR marker:", agmsdr["marker_call"])
print("Restarted STM marker:", restart_stm["marker_call"])
print("Clipped-GD marker:", clipped_gd["marker_call"])


In [ ]:
def draw_trace(x, trace, color, linestyle, label):
    gaps = np.maximum(trace["gaps"][:len(x)], y_floor)
    plt.plot(x[:len(gaps)], gaps, color=color, linestyle=linestyle, linewidth=3.0, label=label)
    if trace["marker_call"] is not None:
        marker_x = trace["marker_call"] + 1
        marker_y = max(trace["marker_gap"], y_floor)
        plt.scatter([marker_x], [marker_y], marker="x", color=color, s=160, linewidths=3.0, zorder=10)


x = np.arange(budget + 1) + 1

clear_output(wait=True)
with sns.plotting_context("notebook"), sns.axes_style("darkgrid"):
    plt.figure(figsize=(28, 6), dpi=120)
    draw_trace(x, agmsdr, "black", "-.", "Projected Restarted AGMsDR")
    draw_trace(x, restart_stm, "#0072B2", "-.", "Restarted STM (Gorbunov et al.)")
    draw_trace(x, clipped_gd, "#E69F00", "--", r"Clipped-GD ($\kappa$-step)")
    plt.axhline(delta ** 2 / mu, color="#666666", linestyle=":", linewidth=2.5, label=r"$\delta^2/\mu$ scale")
    plt.title(r"Regularized logistic regression on a9a, deterministic absolute noise $\delta=10^{-4}$")
    plt.xlabel("noisy gradient oracle calls")
    plt.ylabel(r"$f(x^k)-f^*$")
    plt.xscale("log")
    plt.yscale("log")
    plt.xlim(1, budget + 1)
    plt.ylim(bottom=y_floor)
    plt.legend(fontsize=16, loc="lower left")
    plt.show()
